<img src="images/m-rainbow.svg" width="5%" height="5%">

<h1 style="font-size: 30px; font-weight: bold; color: #ff2f05;">
  The Mistral AI Python SDK
</h1>

The Mistral AI Python SDK (**S**oftware **D**evelopment **K**it) is a wrapper for the **Mistral AI API**.

You can find the official documentation and some examples in:
- The [Vibe Studio Product Section](https://docs.mistral.ai/studio-api/overview) 
- The [API reference](https://docs.mistral.ai/api)
- The [Developers Section](https://docs.mistral.ai/developers)
- Their Github [Python SDK](https://github.com/mistralai/client-python) and [Cookbook](https://github.com/mistralai/cookbook) repositories
- Their [YouTube Streams](https://www.youtube.com/@MistralAIOfficial/streams)

In [ ]:
%pip install mistralai python-dotenv

<h2 style="font-size: 25px; font-weight: bold; color: #fb6227;">
  5. Functions Calling / Tools
</h2>

**Objectives**: create a simple agent, which has access to tools to solve problems.

**Difficulty**: 🟢 ⚪️ ⚪️ ⚪️ - Only some basic Python knowledge is required (imports, lists, dict, loops, etc). 

---
<h3 style="font-size: 20px; font-weight: bold; color: #ff8f1e;">
  5.1 Function Calling
</h3>


LLMs have critical constraints:
- ❌ **Knowledge cutoff date** (no real-time info)
- ❌ **No internal data access**
- ❌ **Cannot perform external actions**


**Solution: Tools**
Give LLMs access to **Python functions** to overcome these limits.

**How It Works**

1. **Expose tools** to the LLM via descriptions
2. **Model responds** with tool calls (no content, just calls):
   ```python
   {
       "tool_calls": [
           {"id": "call_1", "type": "function", "function": {"name": "get_weather", "arguments": "{...}"}}
       ]
   }
   ```
3. **You execute** the functions and add outputs as Tool Messages
4. **Re-invoke the model** → it decides if it can respond or needs more tools

<div style="background-color: #FFF3E0; padding: 12px; border-left: 4px solid #FF8F00; margin: 15px 0; border-radius: 4px;">
  <strong style="color: #9F521A;">💡 Key Insight:</strong>
  Think of a tool as a <strong>documented function exposed to the model</strong> — the LLM can request its use, but <strong>you control execution</strong>.
</div>

---


In [2]:
from mistralai.client import Mistral
from mistralai.client.models import UserMessage
from dotenv import load_dotenv
import os

mistral = Mistral(api_key=os.environ["MISTRAL_API_KEY"])

In [3]:
# We define 2 simple Python functions
def adder(x: int, y: int) -> int:
    return x+y

def multiplier(x: int, y: int) -> int:
    return x*y

In [5]:
# We document these functions using the OpenAI standard (possible to add more arguments such as enum or regex for arguments)
toolbox = [
    {
        "type": "function",
        "function": {
            "name":"adder",
            "description":"Returns the sum of 2 integers",
            "parameters":{
                "type": "object",
                "properties": {
                    "x": {
                        "type": "number",
                        "description": "The first integer to sum-up",
                    },
                    "y": {
                        "type": "number",
                        "description": "The second integer to sum-up",
                    }
                },
                "required": ["x", "y"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name":"multiplier",
            "description":"Returns the product of 2 integers",
            "parameters":{
                "type": "object",
                "properties": {
                    "x": {
                        "type": "number",
                        "description": "The first integer to multiply",
                    },
                    "y": {
                        "type": "number",
                        "description": "The second integer to multiply",
                    }
                },
                "required": ["x", "y"]
            }
        }
    }
]

In [6]:
from mistralai.client.models import SystemMessage

memory = [
    SystemMessage(content="You are a helpful assistant, using tools to reliably respond to questions"),
    UserMessage(content="How much is 3 + 76? Multiply the result by 3.")
]

response = mistral.chat.complete(
    model="mistral-small-latest",
    messages=memory,
    tools=toolbox, # add tools here!
    tool_choice = "any", # auto, any or none
    parallel_tool_calls = False,
    temperature=0
)

In [7]:
# No content
response.choices[0].message.content

''

In [8]:
# But a tool call
response.choices[0].message.tool_calls

[ToolCall(function=FunctionCall(name='adder', arguments='{"x": 3, "y": 76}'), id='V9eWH0GB6', type='function', index=0)]

---

LLMs cannot execute functions directly. Instead, tool calls are requests for you to execute functions on their behalf and return the results.

**Workflow:**

❶ Append the assistant's message (containing tool_calls) to memory<br>
❷ Iterate through each tool call<br>
❸ Extract the function name, arguments, and tool_call_id<br>
❹ Execute the function with the provided arguments<br>
❺ Create a ToolMessage with the output, including the tool_call_id for matching<br>
❻ Re-invoke the LLM after appending all tool messages to memory<br>
❼ Append the new assistant response to memory<br>

<div style="background-color: #FFF3E0; padding: 12px; border-left: 4px solid #FF8F00; margin: 15px 0; border-radius: 4px;">
  <strong style="color: #9F521A;">‼ Note:</strong>
  Sequence matters. The API enforces strict ordering and will error if a ToolMessage follows a UserMessage directly (e.g., if you omitted the initial assistant message).
</div>

---


In [9]:
memory

[SystemMessage(content='You are a helpful assistant, using tools to reliably respond to questions', role='system'),
 UserMessage(content='How much is 3 + 76? Multiply the result by 3.', role='user')]

In [10]:
from mistralai.client.models import ToolMessage
import json

# 1. Add the message to the history
memory.append(response.choices[0].message)


# 2. We use a loop as we are dealing with a list of tool calls (could be several tool calls at once)
for tool_call in response.choices[0].message.tool_calls:

    # 3. Extract info
    _name = tool_call.function.name
    _arguments = json.loads(tool_call.function.arguments)
    _id = tool_call.id

    # 4. Execute the right function (we'll implement a better system later)
    if _name == "adder":
        _output = adder(**_arguments)
    elif _name == "multiplier":
        _output = multiplier(**_arguments)
    
    # 5. Add the output in a tool message and in the conversation history
    memory.append(
        ToolMessage(
            content=str(_output), 
            tool_call_id=_id, 
            name=_name
        )
    )

In [11]:
memory

[SystemMessage(content='You are a helpful assistant, using tools to reliably respond to questions', role='system'),
 UserMessage(content='How much is 3 + 76? Multiply the result by 3.', role='user'),
 AssistantMessage(role='assistant', content='', tool_calls=[ToolCall(function=FunctionCall(name='adder', arguments='{"x": 3, "y": 76}'), id='V9eWH0GB6', type='function', index=0)], prefix=False),
 ToolMessage(content='79', role='tool', tool_call_id='V9eWH0GB6', name='adder')]

In [12]:
# Let's inspect the conversation history
for msg in memory:
    if msg.role=="assistant":
        print(f"{msg.role}: {msg.content} {msg.tool_calls}")
    else:
        print(f"{msg.role}: {msg.content}")

system: You are a helpful assistant, using tools to reliably respond to questions
user: How much is 3 + 76? Multiply the result by 3.
assistant:  [ToolCall(function=FunctionCall(name='adder', arguments='{"x": 3, "y": 76}'), id='V9eWH0GB6', type='function', index=0)]
tool: 79


In [13]:
# 6. Once all tool calls are executed, we can re-invoke the model with the updated conversation history
response = mistral.chat.complete(
    model="mistral-small-latest",
    messages=memory,
    tools=toolbox,
    tool_choice = "auto",
    parallel_tool_calls = False,
    temperature=0
)

# 7. Add the message to the history
memory.append(response.choices[0].message)

In [14]:
# As expected, it still needs another round of tool calls
response.choices[0].message.tool_calls

[ToolCall(function=FunctionCall(name='multiplier', arguments='{"x": 79, "y": 3}'), id='zcDZECTp3', type='function', index=0)]

In [15]:
# Our job now is to extract all paramaters and execute the function accordingly

# We repeat the processing loop
for tool_call in response.choices[0].message.tool_calls:
    _name = tool_call.function.name
    _arguments = json.loads(tool_call.function.arguments)
    _id = tool_call.id

    if _name == "adder":
        _output = adder(**_arguments)
    elif _name == "multiplier":
        _output = multiplier(**_arguments)

    memory.append(
        ToolMessage(
            content=str(_output), 
            tool_call_id=_id, 
            name=_name
        )
    )

# Let's inspect the conversation history
for msg in memory:
    if msg.role=="assistant":
        print(f"{msg.role}: {msg.content} {msg.tool_calls}")
    else:
        print(f"{msg.role}: {msg.content}")

system: You are a helpful assistant, using tools to reliably respond to questions
user: How much is 3 + 76? Multiply the result by 3.
assistant:  [ToolCall(function=FunctionCall(name='adder', arguments='{"x": 3, "y": 76}'), id='V9eWH0GB6', type='function', index=0)]
tool: 79
assistant:  [ToolCall(function=FunctionCall(name='multiplier', arguments='{"x": 79, "y": 3}'), id='zcDZECTp3', type='function', index=0)]
tool: 237


In [17]:
from IPython.display import Markdown

# Once all tool calls are executed, we can re-invoke the model with the updated conversation history
response = mistral.chat.complete(
    model="mistral-small-latest",
    messages=memory,
    tools=toolbox,
    tool_choice = "auto",
    parallel_tool_calls = False,
    temperature=0
)

Markdown(response.choices[0].message.content)

The result of \(3 + 76\) is \(79\), and multiplying this by \(3\) gives **237**.

---
<h3 style="font-size: 20px; font-weight: bold; color: #ff8f1e;">
  5.2 Improving Tool Calls
</h3>

We need to find a way to remove this code block to reduce maintenance:
```python
if _name == "adder":
    _output = adder(**_arguments)
elif _name == "multiplier":
    _output = multiplier(**_arguments)
```

It could be possible to locate the function by name in globals() but this can easily be prone to name conflicts in large projects:
```python
_output = globals[_name](**_arguments)
```

A better approach is to reference the function object directly in the tool documentation.
```python
toolbox = [
    {
        "type": "function",
        "function": {
            "name":"adder",
            "func": adder, # Added a func argument here!
            "description":"Returns the sum of 2 integers",
            "parameters":{

              ...

        }
    }
]
```
---


In [26]:
toolbox = [
    {
        "type": "function",
        "function": {
            "name":"adder",
            "func": adder, # NEW
            "description":"Returns the sum of 2 integers",
            "parameters":{
                "type": "object",
                "properties": {
                    "x": {
                        "type": "number",
                        "description": "The first integer to sum-up",
                    },
                    "y": {
                        "type": "number",
                        "description": "The second integer to sum-up",
                    }
                },
                "required": ["x", "y"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name":"multiplier",
            "func": multiplier, # NEW
            "description":"Returns the product of 2 integers",
            "parameters":{
                "type": "object",
                "properties": {
                    "x": {
                        "type": "number",
                        "description": "The first integer to multiply",
                    },
                    "y": {
                        "type": "number",
                        "description": "The second integer to multiply",
                    }
                },
                "required": ["x", "y"]
            }
        }
    }
]

In [27]:
# Now, we can dynamically create a tools registry
tools_registry = {tool["function"]["name"]: tool["function"]["func"] for tool in toolbox}
tools_registry

{'adder': <function __main__.adder(x: int, y: int) -> int>,
 'multiplier': <function __main__.multiplier(x: int, y: int) -> int>}

In [28]:
# We also define a function to not repeat ourselves
def invoke(user_query: str, memory: list) -> list:

    model = "mistral-small-latest"
    
    # Append system message if this is a new list
    if not memory:
        memory.append(
            SystemMessage(
                content="You are a helpful assistant, using tools to reliably respond to questions"
            )
        )

    # Append user message
    memory.append(UserMessage(content=user_query))

    max_recursion = 10
    iter_index = 0
    while iter_index <= max_recursion:

        # Invoke model
        response = mistral.chat.complete(
            model=model,
            messages=memory,
            tools=toolbox
        )
        
        assistant_msg = response.choices[0].message
        memory.append(assistant_msg)

        if not assistant_msg.tool_calls:
            return memory

        # Execute tools
        for tool_call in assistant_msg.tool_calls:
            _name = tool_call.function.name
            _arguments = json.loads(tool_call.function.arguments)
            _id = tool_call.id

            if _name in tools_registry:
                _output = tools_registry[_name](**_arguments)
                memory.append(
                    ToolMessage(
                        content=str(_output),
                        tool_call_id=_id,
                        name=_name
                    )
                )

        iter_index += 1

    # If we hit max recursion, force one final invocation without tools
    response = mistral.chat.complete(
        model=model,
        messages=memory
    )
    final_msg = response.choices[0].message
    memory.append(final_msg)
    return memory

In [29]:
memory = []
memory = invoke(user_query="How much us 7+9? What about 5+32? Sum both results together", memory=memory)

In [30]:
memory

[SystemMessage(content='You are a helpful assistant, using tools to reliably respond to questions', role='system'),
 UserMessage(content='How much us 7+9? What about 5+32? Sum both results together', role='user'),
 AssistantMessage(role='assistant', content='', tool_calls=[ToolCall(function=FunctionCall(name='adder', arguments='{"x": 7, "y": 9}'), id='SCvXqcXiY', type='function', index=0), ToolCall(function=FunctionCall(name='adder', arguments='{"x": 5, "y": 32}'), id='Di9SgsR26', type='function', index=1)], prefix=False),
 ToolMessage(content='16', role='tool', tool_call_id='SCvXqcXiY', name='adder'),
 ToolMessage(content='37', role='tool', tool_call_id='Di9SgsR26', name='adder'),
 AssistantMessage(role='assistant', content='', tool_calls=[ToolCall(function=FunctionCall(name='adder', arguments='{"x": 16, "y": 37}'), id='sI160lAiv', type='function', index=0)], prefix=False),
 ToolMessage(content='53', role='tool', tool_call_id='sI160lAiv', name='adder'),
 AssistantMessage(role='assista